<h1>TODO: add a column for the cached path of the images or just use the index of the row as the basis</h1>

In [ ]:
import pandas as pd

train_df = pd.read_csv("../dataframes/train_df_final_optimized.csv")
test_df = pd.read_csv("../dataframes/test_df_final_optimized.csv")
val_df = pd.read_csv("../dataframes/val_df_final_optimized.csv")

In [4]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
CUDA version: 12.8
GPU count: 1
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [5]:
import sys
import torch

print("Python executable:")
print(sys.executable)

print("\nTorch location:")
print(torch.__file__)

print("\nTorch version:")
print(torch.__version__)

Python executable:
c:\Users\Drew\Documents\University\2025-2026\2nd Semester\Thesis\Experiment\.venv\Scripts\python.exe

Torch location:
c:\Users\Drew\Documents\University\2025-2026\2nd Semester\Thesis\Experiment\.venv\Lib\site-packages\torch\__init__.py

Torch version:
2.11.0+cu128


In [ ]:
"""
Dual-Input Transfer Learning Pipeline for Binary Breast Cancer Classification
(CBIS-DDSM: Benign vs Malignant)

Backbones compared: ResNet50, VGG16, EfficientNet-B2, DenseNet121
Each model receives TWO inputs per sample: the full mammogram and the
cropped lesion. Both images pass through their own instance of the same
backbone architecture, the resulting feature vectors are concatenated,
and a shared fully-connected head produces the final 2-class prediction.

Prerequisites (must already exist in memory before running this script):
    train_df, val_df, test_df : pandas.DataFrame
        Columns: ["patient_id", "pathology", "image file path",
                  "cropped image file path", "full image cached path",
                  "cropped image cached path"]

Required packages:
    torch, torchvision, pandas, numpy, pillow, scikit-learn

Run as a script (recommended on Windows, because of multiprocessing
DataLoader workers) with train_df / val_df / test_df already loaded,
e.g. by importing them at the top of this file or running this file's
contents in the same session/notebook where those DataFrames exist.
"""

import os
import copy
import time
import warnings

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")

# --------------------------------------------------------------------------
# Reproducibility
# --------------------------------------------------------------------------
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --------------------------------------------------------------------------
# Global configuration
# --------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 16
NUM_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4 if os.name != "nt" else 0  # avoid multiprocessing issues on Windows
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

MODEL_NAMES = ["resnet50", "vgg16", "efficientnet_b2", "densenet121"]
DISPLAY_NAMES = {
    "resnet50": "ResNet50",
    "vgg16": "VGG16",
    "efficientnet_b2": "EfficientNet-B2",
    "densenet121": "DenseNet121",
}

CLASS_NAMES = ["Benign", "Malignant"]

FULL_PATH_COL = "full image cached path"
CROP_PATH_COL = "cropped image cached path"
LABEL_COL = "pathology"


# --------------------------------------------------------------------------
# Label encoding
# --------------------------------------------------------------------------
def encode_label(raw_label):
    """
    Encode the 'pathology' column into a binary label.
    Handles both:
      - already-numeric labels (0/1, possibly as int, float, numpy int, or string '0'/'1')
      - text labels like 'BENIGN', 'MALIGNANT', 'BENIGN_WITHOUT_CALLBACK'
    0 = Benign, 1 = Malignant
    """
    # Case 1: numeric-like label (int, float, numpy scalar, or numeric string)
    try:
        return int(raw_label)
    except (ValueError, TypeError):
        pass

    # Case 2: text label
    text = str(raw_label).strip().upper()
    if "MALIGNANT" in text:
        return 1
    return 0


# --------------------------------------------------------------------------
# Transforms
# --------------------------------------------------------------------------
train_transform = transforms.Compose(
    [
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

eval_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)


# --------------------------------------------------------------------------
# Dataset
# --------------------------------------------------------------------------
class DualImageDataset(Dataset):
    """
    Custom dataset returning (full_image, cropped_image, label) for each
    dataframe row. Both images are loaded from the same row, preserving
    the full-mammogram <-> cropped-lesion relationship. Images are cached
    PNGs, already RGB and 224x224, so no resizing is performed here.
    """

    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        full_image = Image.open(row[FULL_PATH_COL]).convert("RGB")
        cropped_image = Image.open(row[CROP_PATH_COL]).convert("RGB")

        full_image = self.transform(full_image)
        cropped_image = self.transform(cropped_image)

        label = encode_label(row[LABEL_COL])
        label = torch.tensor(label, dtype=torch.long)

        return full_image, cropped_image, label


# --------------------------------------------------------------------------
# Backbone factory
# --------------------------------------------------------------------------
def build_backbone(name):
    """
    Builds an ImageNet-pretrained feature extractor (classifier head
    removed, global-average-pooled to a flat feature vector) for the
    requested backbone name. Returns (feature_extractor, feature_dim).
    """
    if name == "resnet50":
        weights = models.ResNet50_Weights.IMAGENET1K_V2
        base = models.resnet50(weights=weights)
        # children: conv1..layer4, avgpool, fc -> drop fc, keep avgpool
        modules = list(base.children())[:-1]
        feature_extractor = nn.Sequential(*modules)
        feature_dim = 2048

    elif name == "vgg16":
        weights = models.VGG16_Weights.IMAGENET1K_V1
        base = models.vgg16(weights=weights)
        feature_extractor = nn.Sequential(
            base.features,
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        feature_dim = 512

    elif name == "efficientnet_b2":
        weights = models.EfficientNet_B2_Weights.IMAGENET1K_V1
        base = models.efficientnet_b2(weights=weights)
        feature_extractor = nn.Sequential(
            base.features,
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        feature_dim = 1408

    elif name == "densenet121":
        weights = models.DenseNet121_Weights.IMAGENET1K_V1
        base = models.densenet121(weights=weights)
        feature_extractor = nn.Sequential(
            base.features,
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        feature_dim = 1024

    else:
        raise ValueError(f"Unknown backbone name: {name}")

    return feature_extractor, feature_dim


# --------------------------------------------------------------------------
# Dual-input model
# --------------------------------------------------------------------------
class DualInputModel(nn.Module):
    """
    Generic dual-input architecture:

        full image  -> feature extractor A -\
                                              >-- concat -> FC head -> logits
        cropped image -> feature extractor B-/

    Both feature extractors use the same backbone architecture (specified
    by `backbone_name`) but are separate instances (independent weights),
    so the model can learn different representations for the full
    mammogram versus the cropped lesion.
    """

    def __init__(self, backbone_name, num_classes=2, dropout=0.5):
        super().__init__()
        self.backbone_name = backbone_name

        self.full_extractor, feat_dim_full = build_backbone(backbone_name)
        self.crop_extractor, feat_dim_crop = build_backbone(backbone_name)
        assert feat_dim_full == feat_dim_crop
        combined_dim = feat_dim_full + feat_dim_crop

        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, full_image, cropped_image):
        full_feat = self.full_extractor(full_image)
        full_feat = torch.flatten(full_feat, 1)

        crop_feat = self.crop_extractor(cropped_image)
        crop_feat = torch.flatten(crop_feat, 1)

        combined = torch.cat([full_feat, crop_feat], dim=1)
        logits = self.classifier(combined)
        return logits


# --------------------------------------------------------------------------
# Metric computation helper
# --------------------------------------------------------------------------
def compute_metrics(labels, preds, probs):
    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)
    rec = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = float("nan")
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "roc_auc": auc}


# --------------------------------------------------------------------------
# Training loop (one model)
# --------------------------------------------------------------------------
def train_one_model(model_name, train_loader, val_loader, device):
    print(f"\n{'=' * 80}")
    print(f"Training model: {DISPLAY_NAMES[model_name]}")
    print(f"{'=' * 80}")

    model = DualInputModel(model_name).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.1, patience=3
    )
    use_amp = device.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_val_loss = float("inf")
    best_val_acc = 0.0
    epochs_no_improve = 0
    checkpoint_path = f"best_{model_name}.pth"

    for epoch in range(NUM_EPOCHS):
        epoch_start = time.time()

        # ---------------- Training ----------------
        model.train()
        running_loss = 0.0
        train_preds, train_labels = [], []

        for full_imgs, crop_imgs, labels in train_loader:
            full_imgs = full_imgs.to(device, non_blocking=True)
            crop_imgs = crop_imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()

            with torch.cuda.amp.autocast(enabled=use_amp):
                outputs = model(full_imgs, crop_imgs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * labels.size(0)
            preds = torch.argmax(outputs, dim=1)
            train_preds.extend(preds.detach().cpu().numpy())
            train_labels.extend(labels.detach().cpu().numpy())

        train_loss = running_loss / len(train_loader.dataset)
        train_acc = accuracy_score(train_labels, train_preds)

        # ---------------- Validation ----------------
        model.eval()
        val_running_loss = 0.0
        val_preds, val_labels, val_probs = [], [], []

        with torch.no_grad():
            for full_imgs, crop_imgs, labels in val_loader:
                full_imgs = full_imgs.to(device, non_blocking=True)
                crop_imgs = crop_imgs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                with torch.cuda.amp.autocast(enabled=use_amp):
                    outputs = model(full_imgs, crop_imgs)
                    loss = criterion(outputs, labels)

                val_running_loss += loss.item() * labels.size(0)
                probs = torch.softmax(outputs, dim=1)[:, 1]
                preds = torch.argmax(outputs, dim=1)

                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
                val_probs.extend(probs.cpu().numpy())

        val_loss = val_running_loss / len(val_loader.dataset)
        val_metrics = compute_metrics(val_labels, val_preds, val_probs)

        scheduler.step(val_loss)

        epoch_time = time.time() - epoch_start
        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch + 1:3d}/{NUM_EPOCHS} | "
            f"train_loss: {train_loss:.4f} | train_acc: {train_acc:.4f} | "
            f"val_loss: {val_loss:.4f} | val_acc: {val_metrics['accuracy']:.4f} | "
            f"val_prec: {val_metrics['precision']:.4f} | val_rec: {val_metrics['recall']:.4f} | "
            f"val_f1: {val_metrics['f1']:.4f} | val_auc: {val_metrics['roc_auc']:.4f} | "
            f"lr: {current_lr:.2e} | time: {epoch_time:.1f}s"
        )

        # ---------------- Checkpointing / early stopping ----------------
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_metrics["accuracy"]
            epochs_no_improve = 0

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "val_accuracy": val_metrics["accuracy"],
                    "val_loss": val_loss,
                },
                checkpoint_path,
            )
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
                print(
                    f"Early stopping triggered at epoch {epoch + 1} "
                    f"(no improvement for {EARLY_STOPPING_PATIENCE} epochs)."
                )
                break

    # Reload best checkpoint before returning/testing
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])

    print(
        f"Best checkpoint for {DISPLAY_NAMES[model_name]}: "
        f"epoch {checkpoint['epoch'] + 1}, "
        f"val_loss={checkpoint['val_loss']:.4f}, "
        f"val_accuracy={checkpoint['val_accuracy']:.4f}"
    )

    return model, checkpoint


# --------------------------------------------------------------------------
# Test-set evaluation
# --------------------------------------------------------------------------
def evaluate_model(model, test_loader, device, model_name):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for full_imgs, crop_imgs, labels in test_loader:
            full_imgs = full_imgs.to(device, non_blocking=True)
            crop_imgs = crop_imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(full_imgs, crop_imgs)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    metrics = compute_metrics(all_labels, all_preds, all_probs)
    cm = confusion_matrix(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES, zero_division=0)

    print(f"\n--- Test results: {DISPLAY_NAMES[model_name]} ---")
    print(f"Accuracy : {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall   : {metrics['recall']:.4f}")
    print(f"F1-score : {metrics['f1']:.4f}")
    print(f"ROC-AUC  : {metrics['roc_auc']:.4f}")
    print("Confusion Matrix:")
    print(cm)
    print("Classification Report:")
    print(report)

    metrics["confusion_matrix"] = cm
    metrics["classification_report"] = report
    return metrics


# --------------------------------------------------------------------------
# Main pipeline
# --------------------------------------------------------------------------
def run_pipeline(train_df, val_df, test_df):
    print(f"Using device: {DEVICE}")

    train_dataset = DualImageDataset(train_df, train_transform)
    val_dataset = DualImageDataset(val_df, eval_transform)
    test_dataset = DualImageDataset(test_df, eval_transform)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    all_test_results = {}

    for model_name in MODEL_NAMES:
        trained_model, _ = train_one_model(model_name, train_loader, val_loader, DEVICE)
        test_metrics = evaluate_model(trained_model, test_loader, DEVICE, model_name)
        all_test_results[model_name] = test_metrics

        # Free GPU memory before moving to the next backbone
        del trained_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ---------------- Final comparison table ----------------
    print(f"\n{'=' * 88}")
    print("FINAL MODEL COMPARISON (Test Set)")
    print(f"{'=' * 88}")
    header = f"{'Model':<20}{'Accuracy':<12}{'Precision':<13}{'Recall':<10}{'F1-score':<12}{'ROC-AUC':<10}"
    print(header)
    print("-" * len(header))
    for model_name in MODEL_NAMES:
        m = all_test_results[model_name]
        print(
            f"{DISPLAY_NAMES[model_name]:<20}"
            f"{m['accuracy']:<12.4f}"
            f"{m['precision']:<13.4f}"
            f"{m['recall']:<10.4f}"
            f"{m['f1']:<12.4f}"
            f"{m['roc_auc']:<10.4f}"
        )

    best_model_name = max(all_test_results, key=lambda k: all_test_results[k]["roc_auc"])
    print(
        f"\nBest-performing model (by test ROC-AUC): "
        f"{DISPLAY_NAMES[best_model_name]} "
        f"(ROC-AUC = {all_test_results[best_model_name]['roc_auc']:.4f})"
    )

    return all_test_results


In [8]:
print(train_df["pathology"].value_counts())
print(val_df["pathology"].value_counts())
print(test_df["pathology"].value_counts())

pathology
0    1536
1    1063
Name: count, dtype: int64
pathology
0    195
1    133
Name: count, dtype: int64
pathology
0    188
1    138
Name: count, dtype: int64


In [11]:
labels_check = [encode_label(l) for l in val_df["pathology"]]
print(np.bincount(labels_check))  # should now print [195 133]

[195 133]


In [12]:
# --------------------------------------------------------------------------
# Entry point
# --------------------------------------------------------------------------
if __name__ == "__main__":
    # train_df, val_df, and test_df must already be defined in this session
    # (e.g. loaded earlier in the same script/notebook) before running this.
    results = run_pipeline(train_df, val_df, test_df)

Using device: cuda

Training model: ResNet50
Epoch   1/100 | train_loss: 0.6323 | train_acc: 0.6275 | val_loss: 0.5577 | val_acc: 0.7073 | val_prec: 0.6947 | val_rec: 0.4962 | val_f1: 0.5789 | val_auc: 0.7810 | lr: 1.00e-04 | time: 41.6s
Epoch   2/100 | train_loss: 0.5499 | train_acc: 0.7126 | val_loss: 0.5106 | val_acc: 0.7530 | val_prec: 0.7549 | val_rec: 0.5789 | val_f1: 0.6553 | val_auc: 0.8221 | lr: 1.00e-04 | time: 42.2s
Epoch   3/100 | train_loss: 0.4855 | train_acc: 0.7657 | val_loss: 0.6028 | val_acc: 0.7104 | val_prec: 0.7317 | val_rec: 0.4511 | val_f1: 0.5581 | val_auc: 0.8030 | lr: 1.00e-04 | time: 42.7s
Epoch   4/100 | train_loss: 0.4393 | train_acc: 0.7999 | val_loss: 0.5208 | val_acc: 0.7348 | val_prec: 0.6797 | val_rec: 0.6541 | val_f1: 0.6667 | val_auc: 0.8265 | lr: 1.00e-04 | time: 37.2s
Epoch   5/100 | train_loss: 0.4090 | train_acc: 0.8068 | val_loss: 0.5212 | val_acc: 0.7866 | val_prec: 0.7299 | val_rec: 0.7519 | val_f1: 0.7407 | val_auc: 0.8389 | lr: 1.00e-04 | ti